# 05 · AgentTTS baseline — one global configuration (GPU, ~15–20 min)

**Set 1:** `ALLOW_COT = False` (as in the paper). **Set 2b control:** `ALLOW_COT = True` searches the 24-setup table from notebook 04b,
where the answer model can also be "7B/3B with chain-of-thought"; the budgets stay exactly the Set 1 budgets.

The AgentTTS search, ported from the repo (`AgentTTS.py`, same prompts and budget formula), runs on the **dev** rows of the Step 0 table:
initial trials from Insight 1 → written guidelines → new candidates → budget check → dev score, until 8 distinct configurations are tried.
The chosen configuration is then applied unchanged to **every test question** (scores read from the table; the agent never sees test results).

Also reported: the **dev-optimal global config** (exhaustive search of the 16), the ceiling for any single-configuration method.
Runs 3 budgets × 3 seeds, batched together on the GPU. The agent's thinking is saved in `traces/` but never fed back (condition B1).

Output: `runs/agenttts_global/<timestamp>_<GPU>/` (`summary.md`, `agenttts_runs.csv`, `dev_optimal.csv`, `traces/`).

In [1]:
# ==== settings ====
PROJECT_DIR   = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
ALLOW_COT     = True         # False = Set 1 (paper setting); True = Set 2b control (needs notebook 04b)
STEP0_EXPERIMENT = "env_qwen_hotpotqa_cot" if ALLOW_COT else "env_qwen_hotpotqa"
STEP0_RUN_DIR = ""            # empty = latest finished run in runs/<STEP0_EXPERIMENT>; or paste a run folder
EXPERIMENT    = "set2b_agenttts_cot" if ALLOW_COT else "agenttts_global"
AGENT_MODEL_DIR = "Qwen2.5-7B-Instruct"   # agent LLM; "Qwen3-4B" (downloaded in 00) + THINKING = True for thinking mode
THINKING = False
LARGE_MODEL_DIR = "Qwen2.5-7B-Instruct"
BUDGET_QUANTILES = [0.25, 0.5, 1.0]   # budgets = these quantiles of the 16 configs' Eq. 4 costs
SEEDS, ITERATIONS, BATCH_SIZE, MAX_TRIALS = 3, 4, 2, 8
TIME_LIMIT_MIN = 55
RESUME_RUN_DIR = None
DRY_RUN = False

In [2]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

GPU: NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import glob
if not STEP0_RUN_DIR:   # empty = the latest finished results-table run in runs/<STEP0_EXPERIMENT> (dry runs skipped)
    done = [p for p in sorted(glob.glob(f"{PROJECT_DIR}/runs/{STEP0_EXPERIMENT}/*/table/per_question.csv")) if "_dryrun" not in p or DRY_RUN]
    assert done, f"No finished results table in runs/{STEP0_EXPERIMENT} - run notebook {'04b' if 'cot' in STEP0_EXPERIMENT else '04'} first."
    STEP0_RUN_DIR = os.path.dirname(os.path.dirname(done[-1]))
assert os.path.exists(f"{STEP0_RUN_DIR}/table/per_question.csv"), f"{STEP0_RUN_DIR} is not a finished results-table run."
print("Results table:", STEP0_RUN_DIR)
DATA_DIR = json.load(open(f"{STEP0_RUN_DIR}/config.json"))["data_dir"]

Results table: /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/env_qwen_hotpotqa_cot/20260929-140148_A100


In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo
if RESUME_RUN_DIR:
    RUN_DIR = RESUME_RUN_DIR
else:
    stamp = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
    RUN_DIR = f"{PROJECT_DIR}/runs/{EXPERIMENT}/{stamp}_{GPU_TAG}" + ("_dryrun" if DRY_RUN else "")
for sub in ["logs", "code"]:
    os.makedirs(f"{RUN_DIR}/{sub}", exist_ok=True)
print("Run folder:", RUN_DIR)
json.dump({"experiment": EXPERIMENT, "step0_run_dir": STEP0_RUN_DIR, "allow_cot": ALLOW_COT, "agent": AGENT_MODEL_DIR, "thinking": THINKING,
           "budget_quantiles": BUDGET_QUANTILES, "seeds": SEEDS, "iterations": ITERATIONS, "batch_size": BATCH_SIZE,
           "max_trials": MAX_TRIALS, "gpu": gpu, "dry_run": DRY_RUN}, open(f"{RUN_DIR}/config.json", "w"), indent=1)

Run folder: /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/runs/set2b_agenttts_cot/20260929-142812_A100


In [6]:
BACKEND = "mock" if DRY_RUN else "vllm"
CODE = "/content/ssw.py"
shutil.copy(f"{PROJECT_DIR}/code/ssw.py", CODE)
shutil.copy(CODE, f"{RUN_DIR}/code/ssw.py")          # keep the exact code with the results

def minutes_left():
    return max(1.0, TIME_LIMIT_MIN - (time.time() - T_START) / 60)

def run_stage(args, log_name):
    """Runs one ssw.py command in a fresh process; streams output here and into logs/<log_name>."""
    cmd = [sys.executable, "-u", CODE] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log:
        log.write("\n$ " + " ".join(cmd) + "\n")
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line)
            if any(k in line for k in ("it/s]", "s/it]", "INFO ", "(EngineCore")):
                continue                      # keep the notebook readable; everything is in the log file
            print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = round(time.time() - t0)
    json.dump(TIMES, open(f"{RUN_DIR}/logs/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s, {minutes_left():.0f} min left of the limit")
    if p.returncode != 0:
        raise RuntimeError(f"stage failed - see {RUN_DIR}/logs/{log_name}")
TIMES = {}

In [7]:
# vLLM runs only inside the stage processes, so no runtime restart is needed (~3-5 min)
if not DRY_RUN:
    !pip install -q vllm
    !pip uninstall -y -q torchaudio   # Colab's torchaudio is built for another CUDA version and breaks imports
    chk = subprocess.run([sys.executable, "-c", "from vllm import LLM; import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"],
                         capture_output=True, text=True)
    print(chk.stdout or chk.stderr[-2000:])
    assert chk.returncode == 0, "vLLM does not import - Runtime -> Disconnect and delete runtime, then run again."
TIMES["install_vllm"] = round(time.time() - T_START)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72

In [8]:
LOCAL = "/content/models"
MODELS = {
    "S": {"id": "Qwen/Qwen2.5-3B-Instruct", "dir": "Qwen2.5-3B-Instruct", "params": 3.09e9, "nominal": 3e9},
    "L": {"id": "Qwen/Qwen2.5-7B-Instruct", "dir": LARGE_MODEL_DIR,       "params": 7.62e9, "nominal": 7e9},
}
_t = time.time()
for key in [AGENT_MODEL_DIR]:
    m = MODELS[key] if key in MODELS else {"dir": key}
    src, dst = f"{PROJECT_DIR}/models/{m['dir']}", f"{LOCAL}/{m['dir']}"
    m["path"] = dst
    if DRY_RUN:
        continue
    assert os.path.exists(f"{src}/config.json"), f"{src} missing - run 00_setup_download_qwen7b first."
    if not os.path.exists(f"{dst}/config.json"):
        print("copying", m["dir"], "to local disk ...")
        shutil.copytree(src, dst, dirs_exist_ok=True)
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
TIMES["copy_models"] = round(time.time() - _t)
print(f"models ready in {time.time() - _t:.0f}s")
AGENT_PATH = f"{LOCAL}/{AGENT_MODEL_DIR}"

copying Qwen2.5-7B-Instruct to local disk ...
models ready in 205s


In [9]:
run_stage(["agenttts", "--run_dir", RUN_DIR, "--data_dir", DATA_DIR, "--table_dir", f"{STEP0_RUN_DIR}/table",
           "--agent_path", AGENT_PATH, "--backend", BACKEND, "--budget_quantiles", *BUDGET_QUANTILES,
           "--seeds", SEEDS, "--iterations", ITERATIONS, "--batch_size", BATCH_SIZE, "--max_trials", MAX_TRIALS,
           "--time_limit_min", minutes_left(), "--agent_max_tokens", 4096 if THINKING else 1536]
          + (["--thinking"] if THINKING else []) + (["--allow_cot"] if ALLOW_COT else []), "01_agenttts.log")
print("\nAgentTTS run (picked up automatically by notebook " + ("07" if ALLOW_COT else "06") + "):\n", RUN_DIR)

[06:35:17] AgentTTS | 24 setups (chain-of-thought allowed) | budgets (Eq. 4 units, quantiles [0.25, 0.5, 1.0] of setup costs): [54.5, 85.1, 129.4] | seeds 3
[06:38:51]   round 1: distinct configs tried per run [3, 3, 3, 5, 5, 4, 5, 4, 4]
[06:38:57]   round 2: distinct configs tried per run [3, 3, 3, 5, 5, 5, 6, 4, 4]
[06:39:04]   round 3: distinct configs tried per run [3, 3, 4, 5, 5, 5, 6, 4, 4]
[06:39:10]   round 4: distinct configs tried per run [3, 3, 4, 5, 5, 5, 6, 5, 4]
# AgentTTS global configuration - 20260929-142812_A100

Setups searched: 24 (chain-of-thought answering allowed)

Agent: /content/models/Qwen2.5-7B-Instruct (thinking=False) · 72 calls, 113663 in / 14930 out tokens, 35s · max 8 distinct configs per run


## Dev-optimal global config (exhaustive, the ceiling for any single config)

| budget (Eq.4) | config | meaning | dev EM | test EM | test F1 | test TFLOPs/q | test est. GPU s/q |
|---|---|---|---|---|---|---|---|
| 54 | S1-Sc1 | retrieval 3B x1, answer 3B CoT x1 

In [10]:
from IPython.display import Image, Markdown, display
for f in ["summary.md"]:
    p = f"{RUN_DIR}/{f}"
    if not os.path.exists(p):
        continue
    display(Markdown(open(p).read()) if p.endswith(".md") else Image(p))
TIMES["total_notebook"] = round(time.time() - T_START)
json.dump(TIMES, open(f"{RUN_DIR}/logs/stage_times.json", "w"), indent=1)
print("Wall-clock (s):", TIMES)

# AgentTTS global configuration - 20260929-142812_A100

Setups searched: 24 (chain-of-thought answering allowed)

Agent: /content/models/Qwen2.5-7B-Instruct (thinking=False) · 72 calls, 113663 in / 14930 out tokens, 35s · max 8 distinct configs per run


## Dev-optimal global config (exhaustive, the ceiling for any single config)

| budget (Eq.4) | config | meaning | dev EM | test EM | test F1 | test TFLOPs/q | test est. GPU s/q |
|---|---|---|---|---|---|---|---|
| 54 | S1-Sc1 | retrieval 3B x1, answer 3B CoT x1 | 0.315 | 0.370 | 0.467 | 24.8 | 0.39 |
| 85 | S1-Lc5 | retrieval 3B x1, answer 7B CoT x5 | 0.415 | 0.440 | 0.542 | 43.8 | 0.53 |
| 129 | L1-Lc1 | retrieval 7B x1, answer 7B CoT x1 | 0.485 | 0.515 | 0.640 | 58.8 | 0.64 |

## AgentTTS agent (one row per budget x seed)

| budget | seed | distinct configs tried | chosen | dev EM | test EM | test TFLOPs/q | matches dev-optimal |
|---|---|---|---|---|---|---|---|
| 54 | 0 | 3 | S1-Sc1 | 0.315 | 0.370 | 24.8 | yes |
| 54 | 1 | 3 | S1-Sc1 | 0.315 | 0.370 | 24.8 | yes |
| 54 | 2 | 4 | S1-Sc1 | 0.315 | 0.370 | 24.8 | yes |
| 85 | 0 | 5 | S1-Lc5 | 0.415 | 0.440 | 43.8 | yes |
| 85 | 1 | 5 | S1-Lc5 | 0.415 | 0.440 | 43.8 | yes |
| 85 | 2 | 5 | S1-Lc5 | 0.415 | 0.440 | 43.8 | yes |
| 129 | 0 | 6 | L1-Lc1 | 0.485 | 0.515 | 58.8 | yes |
| 129 | 1 | 5 | L1-Lc1 | 0.485 | 0.515 | 58.8 | yes |
| 129 | 2 | 4 | L1-Lc1 | 0.485 | 0.515 | 58.8 | yes |

Traces (every trial, guidelines and the agent's thinking) are in `traces/`. Test scores are read from the Step 0 table; the agent only ever saw dev scores.


Wall-clock (s): {'install_vllm': 240, 'copy_models': 205, '01_agenttts': 240, 'total_notebook': 685}
